Cell 1: Setup & Configuration
This cell initializes the project, checks for available hardware, and sets up a global configuration dictionary for all key parameters.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
import numpy as np
import os
import cv2
import re
from torch.utils.data import Dataset, DataLoader, random_split
from torch.utils.tensorboard import SummaryWriter
from torch.cuda.amp import GradScaler
from albumentations.pytorch import ToTensorV2
import albumentations as A
from tqdm import tqdm
import matplotlib.pyplot as plt
from transformers import SegformerForSemanticSegmentation
import matplotlib.colors as mcolors

# Hardware & Configuration
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"🚀 Running on: {DEVICE}")

CONFIG = {
    # Input
    'img_height': 480,
    'img_width': 640,
    
    # Heads
    'num_seg_classes': 10,
    'num_det_classes': 91,
    
    # Stereo Geometry
    'max_disp_pixel': 192,
    'backbone_stride': 4,
    'internal_disp_steps': 48, # 192px / stride 4 = 48
    
    # Training
    'batch_size': 6,
    'ACCUMULATION_STEPS': 8,
    'lr': 2e-4,
    'num_epochs': 25,
    'num_workers': 4,
    'save_dir': "./checkpoints"
}
os.makedirs(CONFIG['save_dir'], exist_ok=True)

print("✅ Setup complete. Configuration loaded.")


/home/slarc/miniconda3/envs/stereo_wsl/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🚀 Running on: cuda
✅ Setup complete. Configuration loaded.


Cell 2: Fused Model Architecture
This cell contains the complete, final architecture for the FusedHexapodModel, including the shared MobileNetV3 backbone and all three specialized heads (Stereo, Segmentation, and Detection).

In [2]:
# --- Helper Blocks ---
class ConvMean(nn.Module):
    """ Replaces mean() with 1x1 Conv for NPU compatibility """
    def __init__(self, in_channels):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, 1, 1, bias=False)
        with torch.no_grad():
            self.conv.weight.fill_(1.0 / in_channels)
        self.conv.weight.requires_grad = False
    def forward(self, x): return self.conv(x)

class ResBlock(nn.Module):
    """ Standard ResBlock for Refinement """
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, 1, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(channels, channels, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(channels)
    def forward(self, x):
        return self.relu(x + self.bn2(self.conv2(self.relu(self.bn1(self.conv1(x))))))

# --- Stereo Components ---
class SimpleCorrelation(nn.Module):
    """ Loop-based Correlation + Immediate Reduction (Low Memory) """
    def __init__(self, in_channels, max_disp):
        super().__init__()
        self.D = max_disp
        self.reduce = nn.Sequential(
            nn.Conv2d(in_channels, 16, 1, bias=False),
            nn.BatchNorm2d(16), nn.ReLU(inplace=True)
        )
        self.reducer = ConvMean(16) 

    def forward(self, left, right):
        l, r = self.reduce(left), self.reduce(right)
        cost_stack = []
        for d in range(self.D):
            if d > 0:
                sim = self.reducer(l[:,:,:,d:] * r[:,:,:,:-d])
                cost_stack.append(F.pad(sim, (d, 0, 0, 0)))
            else:
                cost_stack.append(self.reducer(l * r))
        return torch.cat(cost_stack, dim=1)

class GaussianGating(nn.Module):
    """ Gate = exp(-0.5 * (dist/sigma)^2) """
    def __init__(self, max_disp, gate_range=12):
        super().__init__()
        self.sigma = gate_range / 2.0
        self.disp_coords = nn.Parameter(
            torch.arange(max_disp).float().view(1, max_disp, 1, 1), 
            requires_grad=False
        )

    def forward(self, cost_volume, coarse_disp_s4):
        dist = self.disp_coords - coarse_disp_s4
        return cost_volume * torch.exp(-0.5 * (dist / self.sigma)**2)

class StereoHeadGatedV2(nn.Module):
    def __init__(self, ch_s16, ch_s4, max_disp_s4):
        super().__init__()
        self.D = max_disp_s4
        self.D_coarse = self.D // 4
        
        # Coarse Stage
        self.corr_coarse = SimpleCorrelation(ch_s16, self.D_coarse)
        self.refine_coarse = nn.Sequential(
            nn.Conv2d(self.D_coarse, 32, 3, 1, 1, bias=False), nn.BatchNorm2d(32), nn.ReLU(),
            ResBlock(32),
            nn.Conv2d(32, self.D_coarse, 3, 1, 1, bias=False)
        )
        self.coarse_sum = nn.Conv2d(self.D_coarse, 1, 1, bias=False) 
        with torch.no_grad():
            self.coarse_sum.weight.data = torch.arange(self.D_coarse).float().view(1, -1, 1, 1)
        self.scale_bias = nn.Parameter(torch.zeros(1, 1, 1, 1))

        # Fine Stage
        self.corr_fine = SimpleCorrelation(ch_s4, self.D)
        self.gating = GaussianGating(max_disp=self.D)
        self.refine_fine = nn.Sequential(
            nn.Conv2d(self.D, 32, 3, 1, 1, bias=False), nn.BatchNorm2d(32), nn.ReLU(),
            ResBlock(32),
            nn.Conv2d(32, self.D, 3, 1, 1, bias=False) 
        )

    def forward(self, l16, r16, l4, r4):
        c_cost = self.refine_coarse(self.corr_coarse(l16, r16))
        prob_c = F.softmax(c_cost * 2.0, dim=1)
        d_coarse_low = self.coarse_sum(prob_c)
        d_coarse_s4 = (F.interpolate(d_coarse_low, size=l4.shape[-2:], mode='bilinear', align_corners=False) * 4.0) + self.scale_bias 
        
        gated_cost = self.gating(self.corr_fine(l4, r4), d_coarse_s4)
        final_cost = self.refine_fine(gated_cost)
        
        return c_cost, final_cost, d_coarse_s4

# --- Segmentation & Detection Heads ---
class LRASPPHead(nn.Module):
    def __init__(self, low_ch, high_ch, num_classes):
        super().__init__()
        self.cbr_high = nn.Sequential(
            nn.Conv2d(high_ch, 128, 1, bias=False), nn.BatchNorm2d(128), nn.ReLU(inplace=True)
        )
        self.scale_high = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Conv2d(high_ch, 128, 1, bias=False), nn.Sigmoid()
        )
        self.low_classifier = nn.Conv2d(low_ch, num_classes, 1)
        self.high_classifier = nn.Conv2d(128, num_classes, 1)

    def forward(self, x_low, x_high):
        out = self.cbr_high(x_high) * self.scale_high(x_high)
        out = F.interpolate(out, size=x_low.shape[-2:], mode='bilinear', align_corners=False)
        return self.low_classifier(x_low) + self.high_classifier(out)

class DecoupledHead(nn.Module):
    def __init__(self, in_ch, num_classes):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(in_ch, in_ch, 1, bias=False), nn.BatchNorm2d(in_ch), nn.SiLU(inplace=True)
        )
        self.cls_branch = nn.Sequential(ResBlock(in_ch), ResBlock(in_ch), nn.Conv2d(in_ch, num_classes, 1))
        self.reg_branch = nn.Sequential(ResBlock(in_ch), ResBlock(in_ch), nn.Conv2d(in_ch, 4, 1))

    def forward(self, x):
        x = self.stem(x)
        return self.cls_branch(x), self.reg_branch(x)

class YOLOHead(nn.Module):
    def __init__(self, ch_dims, num_classes):
        super().__init__()
        self.head_s8  = DecoupledHead(ch_dims[1], num_classes)
        self.head_s16 = DecoupledHead(ch_dims[2], num_classes)
        self.head_s32 = DecoupledHead(ch_dims[3], num_classes)

    def forward(self, x_s8, x_s16, x_s32):
        return [self.head_s8(x_s8), self.head_s16(x_s16), self.head_s32(x_s32)]

# --- Main Fused Model ---
class FusedHexapodModel(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.backbone = timm.create_model('mobilenetv3_large_100', pretrained=True, features_only=True, out_indices=(1, 2, 3, 4))
        bb_ch = self.backbone.feature_info.channels() # [24, 40, 112, 960]
        
        self.stereo_head = StereoHeadGatedV2(ch_s16=bb_ch[2], ch_s4=bb_ch[0], max_disp_s4=config['internal_disp_steps'])
        self.seg_head = LRASPPHead(low_ch=bb_ch[0], high_ch=bb_ch[2], num_classes=config['num_seg_classes'])
        self.yolo_head = YOLOHead(ch_dims=bb_ch, num_classes=config['num_det_classes'])

    def forward(self, left, right=None):
        # Student model expects grayscale, repeats to 3 channels for backbone
        x = left.repeat(1, 3, 1, 1) 
        fl = self.backbone(x)
        
        # Stereo Branch (only if right image provided)
        stereo_preds = None
        if right is not None:
            xr = right.repeat(1, 3, 1, 1)
            fr = self.backbone(xr)
            stereo_preds = self.stereo_head(fl[2], fr[2], fl[0], fr[0])
            
        # Other Heads
        seg_preds = self.seg_head(fl[0], fl[2])
        det_preds = self.yolo_head(fl[1], fl[2], fl[3])
        
        # Return in a consistent order for the loss function
        return stereo_preds, seg_preds, det_preds

model = FusedHexapodModel(CONFIG).to(DEVICE)
# Slower momentum (0.01 instead of 0.1) makes BatchNorm more stable with small batches
for m in model.modules():
    if isinstance(m, nn.BatchNorm2d):
        m.momentum = 0.01
print("✅ Fused model architecture cell is ready.")


Unexpected keys (classifier.bias, classifier.weight, conv_head.bias, conv_head.weight) found while loading pretrained weights. This may be expected if model is being adapted.


✅ Fused model architecture cell is ready.


Cell 3: Unified Loss Function
This cell contains the corrected and unified loss function. It properly calculates the stereo loss on the refined output and combines it with segmentation and detection losses.

In [3]:
class SimpleYOLOLoss(nn.Module):
    def __init__(self, num_classes=91):
        super().__init__()
        self.num_classes = num_classes
        
    def _get_targets(self, targets_list, cls_preds, reg_preds):
        batch_size, _, H, W = cls_preds.shape
        device = cls_preds.device
        dtype = cls_preds.dtype 
        
        cls_targets = torch.zeros_like(cls_preds)
        reg_targets = torch.zeros_like(reg_preds)
        reg_weight = torch.zeros((batch_size, 1, H, W), device=device, dtype=dtype)
        
        for b, t in enumerate(targets_list):
            if t is None or t.numel() == 0: continue
            
            t = t[t[:, 1:5].sum(dim=1) > 0] 
            if t.numel() == 0: continue

            gt_cls = t[:, 0].long()
            valid_mask = (gt_cls < self.num_classes)
            if not valid_mask.all():
                t = t[valid_mask]
                gt_cls = gt_cls[valid_mask]
            
            if t.numel() == 0: continue

            gt_x = (t[:, 1] * W).long()
            gt_y = (t[:, 2] * H).long()
            
            gt_w = t[:, 3] 
            gt_h = t[:, 4] 
            
            gt_x = torch.clamp(gt_x, 0, W - 1)
            gt_y = torch.clamp(gt_y, 0, H - 1)

            # Assign Targets with explicit casting
            cls_targets[b, gt_cls, gt_y, gt_x] = torch.tensor(1.0, device=device, dtype=dtype)
            
            reg_targets[b, 0, gt_y, gt_x] = (gt_w / 2.0).to(dtype)
            reg_targets[b, 1, gt_y, gt_x] = (gt_h / 2.0).to(dtype)
            reg_targets[b, 2, gt_y, gt_x] = (gt_w / 2.0).to(dtype)
            reg_targets[b, 3, gt_y, gt_x] = (gt_h / 2.0).to(dtype)
            
            reg_weight[b, 0, gt_y, gt_x] = torch.tensor(1.0, device=device, dtype=dtype)
                
        return cls_targets, reg_targets, reg_weight

    def forward(self, preds, targets):
        cls_p, reg_p = preds
        cls_t, reg_t, reg_wt = self._get_targets(targets, cls_p, reg_p)
        
        loss_cls = F.binary_cross_entropy_with_logits(cls_p, cls_t, reduction='mean') * 20.0
        
        if reg_wt.sum() > 0:
            loss_reg = F.smooth_l1_loss(reg_p, reg_t, reduction='none')
            loss_reg = (loss_reg * reg_wt).sum() / (reg_wt.sum() + 1e-6)
        else:
            loss_reg = torch.tensor(0.0, device=cls_p.device, dtype=cls_p.dtype)
            
        return loss_cls + loss_reg

class FusedHexapodLoss(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.yolo_s8_loss = SimpleYOLOLoss(config['num_det_classes'])
        self.yolo_s16_loss = SimpleYOLOLoss(config['num_det_classes'])
        self.yolo_s32_loss = SimpleYOLOLoss(config['num_det_classes'])
        
        self.w_stereo = 1.0
        self.w_seg = 1.0  
        self.w_yolo = 1.0
        
        self.temp_kd = 4.0
        # Use batchmean to sum over pixels first, we will divide later
        self.kl_div = nn.KLDivLoss(reduction='batchmean', log_target=True)

    def soft_argmax(self, cost_volume):
        if cost_volume is None: return None
        probs = F.softmax(cost_volume, dim=1)
        indices = torch.arange(cost_volume.shape[1], device=cost_volume.device, dtype=cost_volume.dtype).view(1, -1, 1, 1)
        return torch.sum(probs * indices, dim=1, keepdim=True)

    def forward(self, preds, targets, teacher_preds=None):
        stereo_preds, seg_preds, det_preds = preds
        logs = {}
        total_loss = 0.0

        # --- 1. Stereo Loss ---
        if stereo_preds is not None and targets.get('disp') is not None:
            gt_disp = targets['disp']
            if gt_disp.dim() == 3: gt_disp = gt_disp.unsqueeze(1)
            
            mask = (gt_disp > 0) & (gt_disp < CONFIG['max_disp_pixel'])
            
            if mask.sum() > 0:
                fine_cost_volume = stereo_preds[1]
                pred_disp_s4 = self.soft_argmax(fine_cost_volume)
                pred_disp_full = F.interpolate(pred_disp_s4, size=gt_disp.shape[-2:], mode='bilinear', align_corners=False) * CONFIG['backbone_stride']
                
                l_stereo = F.smooth_l1_loss(pred_disp_full[mask], gt_disp[mask])
                total_loss += self.w_stereo * l_stereo
                logs['stereo'] = l_stereo.item()

        # --- 2. Segmentation Loss (DISTILLATION) ---
        if seg_preds is not None and teacher_preds is not None and 'seg' in teacher_preds:
            teacher_log_probs = teacher_preds['seg']
            student_log_probs = F.log_softmax(seg_preds, dim=1)
            
            # Match resolution
            if student_log_probs.shape[-2:] != teacher_log_probs.shape[-2:]:
                 student_log_probs = F.interpolate(student_log_probs, size=teacher_log_probs.shape[-2:], mode='bilinear', align_corners=False)

            # Calculate Standard KL (Sums over H*W)
            l_seg_kd = self.kl_div(student_log_probs, teacher_log_probs)
            
            # --- FIX: Normalize by spatial dimensions (H * W) ---
            # 'batchmean' only divides by B. We need to divide by H*W too.
            H, W = student_log_probs.shape[-2:]
            l_seg_kd = l_seg_kd / (H * W)
            
            total_loss += self.w_seg * l_seg_kd
            logs['seg_kd'] = l_seg_kd.item()

        # --- 3. YOLO Loss ---
        if det_preds is not None and targets.get('det') is not None:
            has_boxes = any([t is not None and t.numel() > 0 for t in targets['det']])
            
            if has_boxes:
                gt_det = targets['det']
                l_yolo8  = self.yolo_s8_loss(det_preds[0], gt_det)
                l_yolo16 = self.yolo_s16_loss(det_preds[1], gt_det)
                l_yolo32 = self.yolo_s32_loss(det_preds[2], gt_det)
                
                l_yolo = l_yolo8 + l_yolo16 + l_yolo32
                total_loss += self.w_yolo * l_yolo
                logs['yolo'] = l_yolo.item()
            
        return total_loss, logs

criterion = FusedHexapodLoss(CONFIG).to(DEVICE)
print("✅ Pixel-Normalized loss function cell is ready.")

✅ Pixel-Normalized loss function cell is ready.


Cell 4: Data Parsing Helpers
This cell contains the necessary helper functions to scan the respective dataset directories (FlyingThings3D, TartanAir, and COCO) and create a unified list of file paths and labels. The RealFusedDataset class in the next cell will use these functions.

In [4]:
import glob
from pycocotools.coco import COCO

# Note: The 'pycocotools' library is required for this cell: pip install pycocotools
def parse_ft3d(root):
    """ 
    Scans the FlyingThings3D directory for stereo pairs and disparity maps. 
    Adapted for flattened structure: .../image_clean/left/*.png
    """
    print("   Scanning FlyingThings3D...")
    samples = []
    
    # 1. Define paths based on your specific structure
    # Structure: root/frames_cleanpass/TRAIN/image_clean/left/xxxx.png
    img_dir_l = os.path.join(root, 'frames_cleanpass', 'TRAIN', 'image_clean', 'left')
    
    # Structure: root/disparity/TRAIN/disparity/left/xxxx.pfm
    disp_dir_l = os.path.join(root, 'disparity', 'TRAIN', 'disparity', 'left')
    
    if not os.path.exists(img_dir_l):
        print(f"   [Error] Could not find directory: {img_dir_l}")
        return []
    
    # 2. Grab all PNGs directly from the left folder
    # We use glob to get the full paths
    left_files = sorted(glob.glob(os.path.join(img_dir_l, '*.png')))
    
    if not left_files:
        print(f"   [Warning] Directory found but no .png files inside: {img_dir_l}")
        return []

    # 3. Match with Right Image and Disparity
    for l_path in left_files:
        filename = os.path.basename(l_path)
        
        # Construct Right Image Path
        # Replace the folder name 'left' with 'right'
        # Note: We use string replacement on the path. 
        # Be careful if 'left' appears elsewhere in the path, but usually it's safe here.
        r_path = l_path.replace(os.sep + 'left' + os.sep, os.sep + 'right' + os.sep)
        
        # Construct Disparity Path
        # Disparity has extension .pfm
        d_filename = filename.replace('.png', '.pfm')
        d_path = os.path.join(disp_dir_l, d_filename)
        
        # Verify existence
        if os.path.exists(r_path) and os.path.exists(d_path):
            samples.append({
                'type': 'stereo',
                'source': 'ft3d',
                'l': l_path,
                'r': r_path,
                'd': d_path,
                's': None, # No segmentation for FT3D
                'b': None  # No boxes for FT3D
            })
            
    print(f"   -> Found {len(samples)} FT3D pairs.")
    return samples

def parse_tartan(root):
    """ Scans the TartanAir directory for stereo pairs, depth, and optional segmentation. """
    print("   Scanning TartanAir...")
    samples = []
    # Find all 'image_left' folders, which indicates a valid sequence
    left_folders = glob.glob(os.path.join(root, '**', 'image_left'), recursive=True)
    
    for l_folder in left_folders:
        parent = os.path.dirname(l_folder)
        for f in os.listdir(l_folder):
            if not f.endswith('.png'): continue
            
            l_path = os.path.join(l_folder, f)
            r_path = os.path.join(parent, 'image_right', f.replace('_left', '_right'))
            d_path = os.path.join(parent, 'depth_left', f.replace('.png', '_depth.npy'))
            s_path = os.path.join(parent, 'seg_left', f.replace('.png', '_seg.npy'))
            
            # Ensure critical files exist
            if os.path.exists(r_path) and os.path.exists(d_path):
                samples.append({
                    'type': 'stereo', 'source': 'tartan',
                    'l': l_path, 'r': r_path, 'd': d_path, 
                    's': s_path if os.path.exists(s_path) else None, 
                    'b': None
                })
    print(f"   -> Found {len(samples)} TartanAir samples.")
    return samples

def parse_coco(root):
    """ Scans the COCO directory for images and bounding box annotations. """
    print("   Scanning COCO 2017...")
    samples = []
    ann_file = os.path.join(root, 'annotations', 'instances_train2017.json')
    img_dir = os.path.join(root, 'train2017')
    if not os.path.exists(ann_file): return []
    
    coco = COCO(ann_file)
    for iid in coco.getImgIds():
        img_info = coco.loadImgs(iid)[0]
        path = os.path.join(img_dir, img_info['file_name'])
        if not os.path.exists(path): continue
        
        anns = coco.loadAnns(coco.getAnnIds(imgIds=iid, iscrowd=False))
        boxes = []
        for ann in anns:
            x, y, w, h = ann['bbox']
            H, W = img_info['height'], img_info['width']
            # Convert to YOLO format [class, x_center, y_center, width, height]
            boxes.append([ann['category_id'] - 1, (x + w/2)/W, (y + h/2)/H, w/W, h/H])
            
        if boxes:
            samples.append({'type':'mono','source':'coco','l':path,'b':np.array(boxes, dtype=np.float32),'r':None,'d':None,'s':None})
            
    print(f"   -> Found {len(samples)} COCO samples.")
    return samples

print("✅ Data parsing helper functions are ready.")


✅ Data parsing helper functions are ready.


Cell 5: Data Loading & Augmentation
This cell defines the RealFusedDataset class. It uses the parsers from the previous cell to build a master file list and then applies appropriate augmentations for training. It also includes the corrected .pfm file reader.

In [5]:
def read_pfm_fixed(file_path):
    """ Reads a .pfm file and returns a numpy array. Includes scaling. """
    with open(file_path, 'rb') as f:
        header = f.readline().decode().rstrip()
        color = (header == 'PF')
        dim_match = re.match(r'^(\d+)\s(\d+)\s$', f.readline().decode('utf-8'))
        width, height = map(int, dim_match.groups())
        scale = float(f.readline().decode().rstrip())
        endian = '<' if scale < 0 else '>'
        scale = abs(scale)
        
        data = np.fromfile(f, endian + 'f')
        shape = (height, width, 3) if color else (height, width)
        data = np.reshape(data, shape)
        data = np.flipud(data) # PFM files are stored upside down
    return (data * scale).copy()

class RealFusedDataset(Dataset):
    def __init__(self, roots, mode='train', img_size=(480, 640)):
        self.img_h, self.img_w = img_size
        self.samples = []
        
        if 'ft3d' in roots and os.path.exists(roots['ft3d']): self.samples.extend(parse_ft3d(roots['ft3d']))
        if 'tartan' in roots and os.path.exists(roots['tartan']): self.samples.extend(parse_tartan(roots['tartan']))
        if 'coco' in roots and os.path.exists(roots['coco']): self.samples.extend(parse_coco(roots['coco']))
        
        print(f"📊 Total Samples Found: {len(self.samples)}")
        
        # Augmentations for the main (left) image and its targets
        self.transform_main = A.Compose([
            A.Resize(height=self.img_h, width=self.img_w),
            A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
            ToTensorV2()
        ], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels'], min_visibility=0.1))
        
        # Simpler augmentation for the right image (no targets)
        self.transform_right = A.Compose([
            A.Resize(height=self.img_h, width=self.img_w),
            A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
            ToTensorV2()
        ])

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        img_l_rgb = cv2.cvtColor(cv2.imread(s['l']), cv2.COLOR_BGR2RGB)
        h, w, _ = img_l_rgb.shape

        # Load targets
        disp = read_pfm_fixed(s['d']) if s.get('d') and s['d'].endswith('.pfm') else (np.load(s['d']) if s.get('d') else np.full((h, w), -1.0, dtype=np.float32))
        seg = np.load(s['s']) if s.get('s') else np.full((h, w), 255, dtype=np.uint8)
        
        # --- BOX CLEANING LOGIC ---
        boxes, class_labels = [], []
        if s.get('b') is not None and len(s['b']) > 0:
            raw_boxes = np.array(s['b']) # [class, x_c, y_c, w, h]
            
            # 1. Filter out zero-width or zero-height boxes (causes Albumentations crash)
            # YOLO format is [x_center, y_center, width, height]
            valid_mask = (raw_boxes[:, 3] > 1e-5) & (raw_boxes[:, 4] > 1e-5)
            clean_boxes = raw_boxes[valid_mask]
            
            if len(clean_boxes) > 0:
                # 2. Clip coordinates to [0, 1] to prevent out-of-bounds errors
                clean_boxes[:, 1:] = np.clip(clean_boxes[:, 1:], 0.0, 1.0)
                
                # 3. Unzip into separate lists for Albumentations
                # Albumentations expects [bboxes] and [class_labels] separately
                class_labels = clean_boxes[:, 0].tolist()
                boxes = clean_boxes[:, 1:].tolist()

        # Apply main transform
        transformed = self.transform_main(image=img_l_rgb, masks=[disp, seg], bboxes=boxes, class_labels=class_labels)
        
        teacher_input = transformed['image']
        t_disp = transformed['masks'][0].unsqueeze(0)
        t_seg = transformed['masks'][1].long()
        
        # Convert grayscale for the robot model input
        robot_input_l = (teacher_input[0]*0.299 + teacher_input[1]*0.587 + teacher_input[2]*0.114).unsqueeze(0)

        # Handle detection boxes (re-stacking into [class, x, y, w, h])
        if transformed['bboxes']:
            t_det = torch.tensor([[cl] + list(b) for cl, b in zip(transformed['class_labels'], transformed['bboxes'])])
        else:
            t_det = torch.zeros((0, 5))

        # Handle right image
        if s['type'] == 'stereo':
            img_r_rgb = cv2.cvtColor(cv2.imread(s['r']), cv2.COLOR_BGR2RGB)
            t_r = self.transform_right(image=img_r_rgb)['image']
            robot_input_r = (t_r[0] * 0.299 + t_r[1] * 0.587 + t_r[2] * 0.114).unsqueeze(0)
        else:
            robot_input_r = robot_input_l.clone()

        return {
            'left': robot_input_l, 
            'right': robot_input_r, 
            'teacher': teacher_input, 
            'disp': t_disp, 
            'seg': t_seg, 
            'det': t_det
        }

print("✅ RealFusedDataset is ready.")


✅ RealFusedDataset is ready.


Cell6: The Hexapod Visualizer
This function takes a batch and the model predictions, then displays the Left Image, Stereo Disparity, Semantic Map (Hexapod Physics), and YOLO Detections side-by-side.

In [6]:
import matplotlib.colors as mcolors

def visualize_hexapod_output(step, writer):
    model.eval()
    with torch.no_grad():
        # 2 Rows, 4 Columns
        fig, axes = plt.subplots(2, 4, figsize=(24, 12))
        
        for row, (name, b) in enumerate(static_batches.items()):
            l, r = b['left'].to(DEVICE), b['right'].to(DEVICE)
            t_img = b['teacher'].to(DEVICE)
            
            preds = model(l, r)
            stereo, seg, det = preds
            
            student_physics = torch.argmax(seg, dim=1)[0].cpu().numpy()
            teacher_physics = torch.argmax(teacher_seg(t_img), dim=1)[0].cpu().numpy()

            # --- COLUMN 1: GT / INPUT ---
            if name == 'yolo':
                img_bg = l[0, 0].cpu().numpy()
                axes[row, 0].imshow(img_bg, cmap='gray')
                h, w = img_bg.shape
                for box in b['det'][0]:
                    cls, xc, yc, bw, bh = box.tolist()
                    x1, y1 = (xc - bw/2)*w, (yc - bh/2)*h
                    rect = plt.Rectangle((x1, y1), bw*w, bh*h, fill=False, color='lime', linewidth=2)
                    axes[row, 0].add_patch(rect)
                axes[row, 0].set_title(f"GT YOLO BOXES")
            else:
                gt_disp = b['disp'][0, 0].cpu().numpy()
                # LogNorm handles the 'blue crushing' by stretching the range
                # We add a tiny epsilon (1e-3) to avoid log(0)
                vmax_ref = np.percentile(gt_disp[gt_disp > 0], 99) if (gt_disp > 0).any() else 48
                axes[row, 0].imshow(gt_disp, cmap='jet', norm=mcolors.LogNorm(vmin=0.5, vmax=vmax_ref))
                axes[row, 0].set_title(f"GT DISPARITY (LOG-SCALED)")

            # --- COLUMN 2 & 3: SWAPPED FOR NEIGHBORHOOD ---
            if name == 'yolo':
                # ROW 1: Physics | YOLO Heatmap
                axes[row, 1].imshow(student_physics, cmap='tab10', vmin=0, vmax=9)
                axes[row, 1].set_title("STUDENT PHYSICS")
                
                heat = torch.sigmoid(det[1][0][0]).max(dim=0)[0].cpu().numpy()
                axes[row, 2].imshow(heat, cmap='viridis', vmin=0, vmax=1)
                axes[row, 2].set_title("YOLO HEATMAP")
            else:
                # ROW 2: PRED DISPARITY | STUDENT PHYSICS (SWAPPED)
                pred_disp = criterion.soft_argmax(stereo[1])[0, 0].cpu().numpy()
                # Use LogNorm here too for consistency and detail
                axes[row, 1].imshow(pred_disp, cmap='jet', norm=mcolors.LogNorm(vmin=0.5, vmax=vmax_ref))
                axes[row, 1].set_title("PRED DISPARITY (LOG-SCALED)")

                axes[row, 2].imshow(student_physics, cmap='tab10', vmin=0, vmax=9)
                axes[row, 2].set_title("STUDENT PHYSICS")

            # --- COLUMN 4: PRED HITS / TEACHER REF ---
            if name == 'yolo':
                img_bg = l[0, 0].cpu().numpy()
                h, w = img_bg.shape
                axes[row, 3].imshow(img_bg, cmap='gray')
                conf_thresh = 0.7 
                mask = torch.sigmoid(det[1][0][0]).max(dim=0)[0] > conf_thresh
                hit_count = 0
                if mask.any():
                    idx = torch.where(mask)
                    ys, xs = idx[0].cpu().numpy() * 16, idx[1].cpu().numpy() * 16
                    hit_count = len(xs)
                    axes[row, 3].scatter(xs, ys, color='red', s=4, alpha=0.9, edgecolors='none')
                axes[row, 3].set_xlim(0, w); axes[row, 3].set_ylim(h, 0)
                axes[row, 3].set_title(f"PRED HITS: {hit_count}")
            else:
                axes[row, 3].imshow(teacher_physics, cmap='tab10', vmin=0, vmax=9)
                axes[row, 3].set_title("TEACHER PHYSICS (TARGET)")

        for ax in axes.flatten(): ax.axis('off')
        plt.tight_layout()
        writer.add_figure('Model_Progress/Deep_Vision_Dashboard', fig, global_step=step)
        plt.close(fig)

print("🎯 Monitoring Dashboard: Neighborhood (Stereo) & Living Room (YOLO) are ready.")

🎯 Monitoring Dashboard: Neighborhood (Stereo) & Living Room (YOLO) are ready.


Cell 7: Model Graph
1. The Shared Backbone: Where the image features are first extracted.
2. The Split: Where the data branches off into three different "heads" (Stereo, Segmentation, and YOLO).
3. The Stereo Neck: How the left and right features are concatenated or subtracted to form a cost volume.
4. Tensor Shapes: Most importantly, it labels the dimensions (e.g., $1 \times 120 \times 160$) at every step, which helps you verify your downsampling logic is working correctly.

In [7]:
# Create a dummy input matching your robot's input size
dummy_left = torch.randn(1, 1, CONFIG['img_height'], CONFIG['img_width']).to(DEVICE)
dummy_right = torch.randn(1, 1, CONFIG['img_height'], CONFIG['img_width']).to(DEVICE)

# Add the graph to TensorBoard
writer = SummaryWriter("./logs")

class GraphWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, l, r):
        stereo, seg, det = self.model(l, r)
        # Flatten everything into one long tuple of tensors
        # det is usually [scale8, scale16, scale32]
        return stereo[0], stereo[1], seg, det[0][0], det[0][1], det[1][0], det[1][1], det[2][0], det[2][1]

# 1. Create the wrapper
wrapper = GraphWrapper(model).to(DEVICE)

# 2. Use the wrapper for the graph
print("📊 Generating model graph for TensorBoard (using wrapper)...")
writer.add_graph(wrapper, [dummy_left, dummy_right])
print("✅ Model graph for TensorBoard done...")
# 3. Clean up memory
del wrapper
torch.cuda.empty_cache()

📊 Generating model graph for TensorBoard (using wrapper)...
✅ Model graph for TensorBoard done...


Cell 8: The Final Training Loop
This cell sets up and executes the main training process. It includes the crucial hexapod_collate function to handle mixed-data batches, initializes the data loaders, optimizer, and scheduler, and contains the complete training and validation logic with logging to TensorBoard.

In [ ]:
# --- Segmentation Teacher Model ---
class SegmentationTeacher(nn.Module):
#class UnifiedHexapodTeacher(nn.Module):
    """
    A 10-Class Teacher that bridges Indoor (ADE20K) and Outdoor (ADE20K/TartanAir)
    semantics for a Hexapod Robot.
    """
    def __init__(self, device='cuda'):
        super().__init__()
        # Using SegFormer B4 for high accuracy on texture details (Grass vs Carpet)
        model_name = "nvidia/segformer-b4-finetuned-ade-512-512"
        self.model = SegformerForSemanticSegmentation.from_pretrained(model_name)
        self.model.eval()
        self.model.to(device)
        
        for param in self.model.parameters():
            param.requires_grad = False

        self.num_source_classes = 150
        self.num_target_classes = 10  # Unified List
        
        map_matrix = torch.zeros(self.num_source_classes, self.num_target_classes)

        # --- 1. HARD_FLAT (Class 0) ---
        # Indoor: Floor(3), Wood(19)
        # Outdoor: Road(6), Sidewalk(11), Path(16), Platform(53), Flooring(105)
        # Physics: Rigid, predictable friction.
        hard_indices = [3, 19, 6, 11, 16, 53, 105]
        map_matrix[hard_indices, 0] = 1.0

        # --- 2. SOFT_FLAT (Class 1) ---
        # Indoor: Rug(29), Carpet(57)
        # Physics: High friction, but generally flat.
        soft_flat_indices = [29, 57]
        map_matrix[soft_flat_indices, 1] = 1.0

        # --- 3. NATURAL_UNEVEN (Class 2) ---
        # Outdoor: Grass(9), Earth(13), Field(28), Sand(46), Soil(94), Land(93), Rock(122 - gravel)
        # Physics: Compliant (sinking feet), uneven height. Needs High-Step.
        natural_indices = [9, 13, 28, 46, 94, 93, 122]
        map_matrix[natural_indices, 2] = 1.0

        # --- 4. WATER (Class 3) ---
        # Water(21), Sea(26), River(60), Lake(103), Pool(128)
        map_matrix[[21, 26, 60, 103, 128], 3] = 1.0

        # --- 5. CLUTTER (Class 4) ---
        # Step-over items: Pillow(59), Box(70), Paper(96), Towel(100), Clothes(106), Bag(112)
        # Note: In TartanAir, this corresponds to small scattering objects.
        clutter_indices = [59, 70, 96, 100, 106, 112]
        map_matrix[clutter_indices, 4] = 1.0

        # --- 6. STAIRS (Class 5) ---
        # Stairs(25), Step(126), Escalator(124)
        map_matrix[[25, 126, 124], 5] = 1.0

        # --- 7. OBSTACLES (Class 6) ---
        # Structural: Wall(0), Building(1), Fence(32), Railing(104)
        # Furniture: Cabinet(10), Bed(7), Chair(19), Sofa(23), Table(15), Shelf(41)
        # Outdoor: Tree(4), Plant(17 - bushes)
        obs_indices = [0, 1, 32, 104, 10, 7, 19, 23, 15, 41, 4, 17]
        map_matrix[obs_indices, 6] = 1.0

        # --- 8. GLASS (Class 7) ---
        # Window(8), Glass(63), Mirror(66)
        map_matrix[[8, 63, 66], 7] = 1.0

        # --- 9. DYNAMIC (Class 8) ---
        # Person(12), Car(20), Bus(80), Bicycle(127)
        map_matrix[[12, 20, 80, 127], 8] = 1.0

        # --- 10. SKY/VOID (Class 9) ---
        # Sky(2), Ceiling(5), Light(82)
        map_matrix[[2, 5, 82], 9] = 1.0

        # Fill remaining unassigned classes to OBSTACLES
        current_assigned = map_matrix.sum(dim=1)
        unassigned_indices = (current_assigned == 0).nonzero(as_tuple=True)[0]
        map_matrix[unassigned_indices, 6] = 1.0 
        
        self.register_buffer('map_matrix', map_matrix)
        self.register_buffer('mean', torch.tensor([0.485, 0.456, 0.406]).view(1,3,1,1))
        self.register_buffer('std',  torch.tensor([0.229, 0.224, 0.225]).view(1,3,1,1))

    def forward(self, x):
        with torch.no_grad():
            mean = self.mean.to(x.device)
            std = self.std.to(x.device)
            map_matrix = self.map_matrix.to(x.device)
            x_norm = (x - mean) / std
            outputs = self.model(x_norm)
            probs_150 = F.softmax(outputs.logits, dim=1).permute(0, 2, 3, 1)
            probs_10 = torch.matmul(probs_150, map_matrix).permute(0, 3, 1, 2)
            
            return torch.log(probs_10 + 1e-6)

# --- Verification Step ---
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# 1. Initialize the teacher and move it to the GPU/CPU
teacher = SegmentationTeacher().to(DEVICE)

# 2. Create a dummy batch of RGB images
# Shape: [Batch Size, Channels, Height, Width]
dummy_rgb_input = torch.rand(2, 3, 480, 640).to(DEVICE)

# 3. Perform a forward pass
teacher_logits = teacher(dummy_rgb_input)

# 4. Check the output shape
print("\n--- Teacher Integration Test ---")
print(f"Input Shape:  {dummy_rgb_input.shape}")
print(f"Output Shape: {teacher_logits.shape}")

# The output shape should be [Batch Size, 40, Height, Width]
# 40 is the number of classes in the NYUv2 dataset.
if teacher_logits.shape == (2, 10, 120, 160):
    print("✅ SUCCESS: The teacher model produced logits with the correct shape.")
else:
    print("❌ FAILURE: The output shape is incorrect. Please review the code.")


# --- Custom Collate Function ---
def hexapod_collate(batch):
    """ Custom collate to handle variable numbers of detection boxes per image. """
    keys = batch[0].keys()
    collated = {k: [d[k] for d in batch] for k in keys}
    
    # Stack tensors that have a fixed size
    for k in ['left', 'right', 'teacher', 'disp', 'seg']:
        collated[k] = torch.stack(collated[k])
    # 'det' remains a list of tensors
    return collated

# The indices we selected
NEIGHBORHOOD_IDX = 34795 
YOLO_IDX = 128923

# Function to pull and prepare a sample
def prepare_static_sample(dataset, idx):
    # Check if the index is in the training set or needs to be pulled from the base
    base_ds = dataset.dataset if hasattr(dataset, 'dataset') else dataset
    item = base_ds[idx]
    return {k: v.unsqueeze(0).to(DEVICE) if isinstance(v, torch.Tensor) else v 
            for k, v in item.items()}







# --- Data Setup ---
DATA_ROOTS = {
    'coco':   '../datasets/coco',
    'ft3d':   '../datasets/FlyingThings3D',
    'tartan': '../datasets/TartanAir'
}
full_dataset = RealFusedDataset(DATA_ROOTS, img_size=(CONFIG['img_height'], CONFIG['img_width']))
train_size = int(0.95 * len(full_dataset))
train_ds, val_ds = random_split(full_dataset, [train_size, len(full_dataset) - train_size])

train_loader = DataLoader(train_ds, batch_size=CONFIG['batch_size'], shuffle=True, num_workers=CONFIG['num_workers'], pin_memory=True, drop_last=True, collate_fn=hexapod_collate)
val_loader = DataLoader(val_ds, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=CONFIG['num_workers'], pin_memory=True, collate_fn=hexapod_collate)

# Save this for the whole training run
static_batches = {
    'yolo': prepare_static_sample(train_ds, YOLO_IDX),
    'neighborhood': prepare_static_sample(train_ds, NEIGHBORHOOD_IDX)
}
    
# --- Training Setup ---
teacher_seg = SegmentationTeacher(device=DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG['lr'], weight_decay=1e-4)


# Calculate total updates
# If len(train_loader) is 21100 and ACCUMULATION_STEPS is 8, 
# you'll take ~2637 updates per epoch.
steps_per_epoch = len(train_loader) // CONFIG['ACCUMULATION_STEPS']

scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=CONFIG['lr'],
    epochs=CONFIG['num_epochs'],
    steps_per_epoch=steps_per_epoch, # <--- CRITICAL FIX
    pct_start=0.1,   # 10% warmup is usually better for distillation
    div_factor=25,   # Start at lr/25
    final_div_factor=1e4
)


scaler = GradScaler()

 
model.train()

def train_one_epoch(epoch_idx):
    model.train()
    optimizer.zero_grad()
    pbar = tqdm(train_loader, desc=f"Epoch {epoch_idx+1}/{CONFIG['num_epochs']}")
    
    for i, batch in enumerate(pbar):
        global_step = epoch_idx * len(train_loader) + i
        
        l, r = batch['left'].to(DEVICE), batch['right'].to(DEVICE)
        t_img = batch['teacher'].to(DEVICE)
        targets = {
            'disp': batch['disp'].to(DEVICE),
            'det': [t.to(DEVICE) for t in batch['det']]
        }

        with torch.no_grad():
            teacher_preds = {'seg': teacher_seg(t_img)}

        with torch.amp.autocast('cuda'):
            preds = model(l, r)
            loss, logs = criterion(preds, targets, teacher_preds)
            distorted_loss = loss / CONFIG['ACCUMULATION_STEPS']

        # Safety Check
        if torch.isnan(loss):
            print(f"⚠️ NaN detected at step {global_step}. Stopping!")
            return False 

        scaler.scale(distorted_loss).backward()

        if (i + 1) % CONFIG['ACCUMULATION_STEPS'] == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            scheduler.step() 

        if i % 10 == 0:
            current_lr = optimizer.param_groups[0]['lr']
            writer.add_scalar('Loss/Total', loss.item(), global_step)
            writer.add_scalar('Params/LearningRate', current_lr, global_step)
            for k, v in logs.items():
                writer.add_scalar(f'Loss/{k}', v, global_step)

        if i % 500 == 0:
            # Calls the dashboard with GT and Teacher comparisons
            visualize_hexapod_output(step=global_step, writer=writer)
            model.train()

        pbar.set_postfix({'loss': f"{loss.item():.2f}", 'lr': f"{optimizer.param_groups[0]['lr']:.1e}"})
    return True
    
# --- Main Execution ---
print("🚀 Starting training...")
for epoch in range(CONFIG['num_epochs']):
    train_one_epoch(epoch)
    # Add validation loop here if needed
    torch.save(model.state_dict(), os.path.join(CONFIG['save_dir'], f"hexapod_epoch_{epoch+1}.pth"))
print("🏁 Training complete.")


/home/slarc/miniconda3/envs/stereo_wsl/lib/python3.10/site-packages/huggingface_hub/file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(



--- Teacher Integration Test ---
Input Shape:  torch.Size([2, 3, 480, 640])
Output Shape: torch.Size([2, 10, 120, 160])
✅ SUCCESS: The teacher model produced logits with the correct shape.
   Scanning FlyingThings3D...
   -> Found 21818 FT3D pairs.
   Scanning TartanAir...
   -> Found 38603 TartanAir samples.
   Scanning COCO 2017...
loading annotations into memory...
Done (t=9.73s)
creating index...
index created!
   -> Found 117266 COCO samples.
📊 Total Samples Found: 177687


/tmp/ipykernel_2954463/1170895027.py:193: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


🚀 Starting training...


Epoch 1/25:   0%|                                | 7/28133 [00:04<2:51:57,  2.73it/s, loss=145.88, lr=8.0e-06]/home/slarc/miniconda3/envs/stereo_wsl/lib/python3.10/site-packages/torch/optim/lr_scheduler.py:224: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  warnings.warn(
Epoch 1/25:   1%|▍                             | 354/28133 [01:44<2:07:52,  3.62it/s, loss=127.03, lr=8.0e-06]

In [ ]:
%abort

In [ ]:
def browse_coco_furniture(dataset, num_to_show=5, offset=0):
    base_ds = dataset.dataset if hasattr(dataset, 'dataset') else dataset
    indices = dataset.indices if hasattr(dataset, 'indices') else range(len(dataset))
    
    found_count = 0
    shown_count = 0
    
    # Common COCO IDs: 56: chair, 60: dining table, 62: tv, 63: laptop
    furniture_ids = {56, 60, 62, 63, 72} 

    print("🕵️ Scanning COCO for indoor scenes with objects...")

    for idx in indices:
        sample_info = base_ds.samples[idx]
        if 'coco' in sample_info['l'].lower():
            batch = base_ds[idx]
            det = batch['det']
            
            # Check if this image has any furniture
            has_furniture = False
            if det.numel() > 0:
                classes = det[:, 0].unique().tolist()
                if any(c in furniture_ids for c in classes):
                    has_furniture = True

            if has_furniture:
                found_count += 1
                if found_count <= offset: continue
                
                img = batch['left'][0].cpu().numpy()
                
                plt.figure(figsize=(8, 6))
                plt.imshow(img, cmap='gray')
                h, w = img.shape
                for box in det:
                    cls_id, xc, yc, bw, bh = box.tolist()
                    x1, y1 = (xc - bw/2) * w, (yc - bh/2) * h
                    rect = plt.Rectangle((x1, y1), bw*w, bh*h, fill=False, color='cyan', linewidth=2)
                    plt.gca().add_patch(rect)
                    plt.text(x1, y1-5, f"ID:{int(cls_id)}", color='cyan', weight='bold')
                
                plt.title(f"COCO Index: {idx} | Objects: {len(det)}")
                plt.axis('off')
                plt.show()

                shown_count += 1
                if shown_count >= num_to_show: break

browse_coco_furniture(train_ds, num_to_show=10, offset=30)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cv2

def find_stairs_visually(dataset, num_to_show=5, offset=0):
    """
    Scans TartanAir for images that 'look like' stairs based on 
    segmentation geometry (repetitive horizontal patterns).
    """
    base_ds = dataset.dataset if hasattr(dataset, 'dataset') else dataset
    indices = dataset.indices if hasattr(dataset, 'indices') else range(len(dataset))
    
    found_count = 0
    shown_count = 0

    print("🪜 Visually scanning TartanAir for stair-like geometry...")

    for idx in indices:
        sample = base_ds.samples[idx]
        if 'tartan' not in sample['l'].lower() or not sample.get('s'):
            continue
            
        # Load raw segmentation
        raw_seg = np.load(sample['s']).astype(np.uint8)
        
        # Calculate horizontal gradients (to find the 'treads' of the stairs)
        # Stairs in TartanAir usually have many horizontal transitions
        grad_y = cv2.Sobel(raw_seg, cv2.CV_64F, 0, 1, ksize=3)
        edge_density = np.sum(np.abs(grad_y) > 0)
        
        # Heuristic: Stairs have high horizontal edge density in a local area
        if edge_density > 15000: # Threshold for 'lot of horizontal lines'
            found_count += 1
            if found_count <= offset: continue
            
            # Load images for confirmation
            img = plt.imread(sample['l'])
            
            plt.figure(figsize=(12, 5))
            plt.subplot(1, 2, 1)
            plt.imshow(img)
            plt.title(f"Potential Stair Scene (Index: {idx})")
            
            plt.subplot(1, 2, 2)
            plt.imshow(raw_seg, cmap='nipy_spectral')
            plt.title("Segmentation Pattern")
            plt.show()
            
            shown_count += 1
            if shown_count >= num_to_show: break
            
    return

# Run the visual hunter
find_stairs_visually(train_ds, num_to_show=10, offset=50551)

In [ ]:
# The unique "fingerprint" of your image
target_file = "neighborhood/Easy/P013/image_left/000473_left.png"

# Handle Subset vs Dataset
base_ds = train_ds.dataset if hasattr(train_ds, 'dataset') else train_ds

# Fuzzy search: check if our target string exists inside the stored path
try:
    NEIGHBORHOOD_IDX = next(
        i for i, s in enumerate(base_ds.samples) 
        if target_file in s['l'].replace('\\', '/')
    )
    print(f"✅ Success! Found Neighborhood sample at Index: {NEIGHBORHOOD_IDX}")
    print(f"📍 Full Path in dataset: {base_ds.samples[NEIGHBORHOOD_IDX]['l']}")
except StopIteration:
    print("❌ Still couldn't find it. Let's print a sample path to see the format:")
    print(f"Sample path from dataset: {base_ds.samples[0]['l']}")

In [ ]:
# Handle Subset vs Dataset
base_ds = train_ds.dataset if hasattr(train_ds, 'dataset') else train_ds

# Find all indices belonging to 'office'
office_indices = [i for i, s in enumerate(base_ds.samples) if 'office' in s['l'].lower()]

if office_indices:
    print(f"✅ Found {len(office_indices)} office samples.")
    print(f"📍 Office Range: {min(office_indices)} to {max(office_indices)}")
    
    # Let's peek at a random one in the middle of the range
    mid_idx = office_indices[len(office_indices)//2]
    print(f"👉 Recommended starting point for scanning: {mid_idx}")
else:
    print("❌ 'office' keyword not found in any file paths. Check your folder names!")